<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/19_final_outputs/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 19: Final Outputs and Agent Completion

In our final lesson about the research agent implementation, we will complete the research agent's workflow by implementing the remaining tools that filter research results, scrape additional sources, and compile everything into a final research file. We'll test the complete end-to-end agent and analyze its output.

Learning Objectives:
- Learn how to filter and validate research sources for quality and trustworthiness
- Understand how to select the most valuable sources for full content scraping
- Implement the final research compilation tool that creates structured outputs
- Test the complete agent workflow and analyze output quality

## 1. Setup

### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and loads your `GOOGLE_API_KEY` from Colab Secrets automatically.

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL/DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [1]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.8",
            "nest-asyncio2",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

    # Load API keys from Colab Secrets
    # In Colab: Secrets tab (key icon) → Add new secret → Name: GOOGLE_API_KEY
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    os.environ["FIRECRAWL_API_KEY"] = userdata.get("FIRECRAWL_API_KEY")
    os.environ["PPLX_API_KEY"] = userdata.get("PPLX_API_KEY")
    try:
        os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")
    except Exception:
        pass  # Optional for public GitHub repositories

In [2]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    from utils import env

    env.load(required_env_vars=["GOOGLE_API_KEY", "PPLX_API_KEY", "FIRECRAWL_API_KEY"])

Environment variables loaded from `/Users/jai/Documents/code-repo/agentic-ai-engineering-course/.env`
Environment variables loaded successfully.


### Import Key Packages

In [3]:
import nest_asyncio2

nest_asyncio2.apply()  # Allow nested async usage in notebooks

### Download the Nova sample data

The notebook downloads a clean, lesson-specific research fixture and copies it into a writable workspace. The default works without editing any paths. To use your own folder, either edit `CUSTOM_RESEARCH_FOLDER` or set the `NOVA_RESEARCH_FOLDER` environment variable before running this cell.

In [ ]:
import os
import shutil
import urllib.request
import zipfile
from pathlib import Path

NOVA_SAMPLES_URL = os.environ.get(
    "NOVA_SAMPLES_URL",
    "https://raw.githubusercontent.com/towardsai/agentic-ai-engineering-course/main/data/nova_samples.zip",
)
LESSON_SAMPLE = "19_final_outputs"

# Option A: edit this line to point at your own folder.
CUSTOM_RESEARCH_FOLDER = None
# Option B: set the NOVA_RESEARCH_FOLDER environment variable instead.
CUSTOM_RESEARCH_FOLDER = CUSTOM_RESEARCH_FOLDER or os.environ.get("NOVA_RESEARCH_FOLDER")

if CUSTOM_RESEARCH_FOLDER:
    research_folder = Path(CUSTOM_RESEARCH_FOLDER).expanduser().resolve()
else:
    archive_path = Path("nova_samples.zip")
    samples_directory = Path("nova_samples")
    research_folder = Path("nova_workspace") / LESSON_SAMPLE

    # Always refresh the downloaded fixtures so notebook reruns cannot use stale data.
    if samples_directory.exists():
        shutil.rmtree(samples_directory)
    archive_path.unlink(missing_ok=True)
    urllib.request.urlretrieve(NOVA_SAMPLES_URL, archive_path)
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(".")
    archive_path.unlink()

    # Nova mutates its research folder, so work on a resettable copy of the pristine fixture.
    if research_folder.exists():
        shutil.rmtree(research_folder)
    research_folder.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(samples_directory / LESSON_SAMPLE, research_folder)
    research_folder = research_folder.resolve()

if not (research_folder / "article_guideline.md").is_file():
    raise FileNotFoundError(f"Expected article_guideline.md in {research_folder}")

print(f"Using research folder: {research_folder}")

## 2. Completing the Research Workflow

As we've seen in previous lessons, our research agent follows a systematic workflow. We've covered the initial data ingestion (lesson 16) and the research loop with query generation (lesson 17). Now we need to implement the final steps that ensure quality and compile the results.

The complete workflow includes these final steps:

```markdown
4. Filter Perplexity results by quality:
    4.1 Run the "select_research_sources_to_keep" tool to automatically evaluate each source 
    for trustworthiness, authority and relevance.

5. Identify which of the accepted sources deserve a full scrape:
    5.1 Run the "select_research_sources_to_scrape" tool to choose up to 5 diverse, 
    authoritative sources whose full content will add most value.
    5.2 Run the "scrape_research_urls" tool to scrape/clean each selected URL's full content.

6. Write final research file:
    6.1 Run the "create_research_file" tool to combine all research data into a 
    comprehensive research.md file.
```

Let's examine each of these final tools and understand their purpose in the workflow.

## 3. Filtering Research Sources for Quality

The `select_research_sources_to_keep` tool addresses a critical problem we discovered during development: Perplexity results often include sources from untrustworthy blogs, SEO spam, or low-quality content that would pollute our research.

### 3.1 Understanding the Tool Implementation

This tool takes a research directory as input and automatically filters Perplexity results for quality. It reads the article guidelines and raw Perplexity results, then uses an LLM to evaluate each source based on trustworthiness, authority, and relevance criteria. The tool outputs two files: a list of selected source IDs and a filtered markdown file containing only the approved sources. This automated filtering saves time while ensuring research quality.

Source: _mcp_server/src/tools/select_research_sources_to_keep_tool.py_

```python
async def select_research_sources_to_keep_tool(research_directory: str) -> Dict[str, Any]:
    """
    Automatically select high-quality sources from Perplexity results.

    Uses an LLM to evaluate each source in perplexity_results.md for trustworthiness,
    authority, and relevance based on the article guidelines. Writes the comma-separated
    IDs of accepted sources to perplexity_sources_selected.md and saves a filtered
    markdown file perplexity_results_selected.md containing only the accepted sources.

    Args:
        research_directory: Path to the research directory containing article guidelines and research data

    Returns:
        Dict with status, selection results, and file paths
    """
    # Convert to Path object
    research_path = Path(research_directory)
    nova_path = research_path / NOVA_FOLDER
    
    # Gather context from the research folder
    guidelines_path = research_path / ARTICLE_GUIDELINE_FILE
    results_path = nova_path / PERPLEXITY_RESULTS_FILE
    
    article_guidelines = read_file_safe(guidelines_path)
    perplexity_results = read_file_safe(results_path)
    
    # Use LLM to select sources
    selected_ids = await select_sources(
        article_guidelines, perplexity_results, settings.source_selection_model
    )
    
    # Write selected source IDs to file
    sources_selected_path = nova_path / PERPLEXITY_SOURCES_SELECTED_FILE
    sources_selected_path.write_text(",".join(map(str, selected_ids)), encoding="utf-8")
    
    # Extract and save filtered content
    filtered_content = extract_selected_blocks_content(selected_ids, perplexity_results)
    results_selected_path = nova_path / PERPLEXITY_RESULTS_SELECTED_FILE
    results_selected_path.write_text(filtered_content, encoding="utf-8")
    
    return {
        "status": "success",
        "sources_selected_count": len(selected_ids),
        "selected_source_ids": selected_ids,
        "sources_selected_path": str(sources_selected_path.resolve()),
        "results_selected_path": str(results_selected_path.resolve()),
        "message": f"Successfully selected {len(selected_ids)} high-quality sources..."
    }
```

The core of this tool is the `select_sources` function, which uses the `PROMPT_SELECT_SOURCES` prompt to evaluate each source and select the most relevant ones.

### 3.2 The Source Evaluation Prompt

Here's the prompt used to evaluate the sources:

Source: _mcp_server/src/config/prompts.py_

```python
PROMPT_SELECT_SOURCES = """
You are a research quality evaluator. Your task is to evaluate web sources for an upcoming article
and select only the high-quality, trustworthy sources that are relevant to the article guidelines.

<article_guidelines>
{article_guidelines}
</article_guidelines>

Here are the sources to evaluate:
<sources_to_evaluate>
{sources_data}
</sources_to_evaluate>

**Selection Criteria:**
- ACCEPT sources from trustworthy domains (e.g., .edu, .gov, established news sites,
official documentation, reputable organizations)
- ACCEPT sources with high-quality, relevant content that directly supports the article guidelines
- REJECT sources from obscure, untrustworthy, or potentially biased websites
- REJECT sources with low-quality, irrelevant, or superficial content
- REJECT sources that seem to be marketing materials, advertisements, or self-promotional content

Return your decision as a structured output with:
1. selection_type: "none" if no sources meet the quality standards, "all" if all sources are acceptable,
or "specific" for specific source IDs
2. source_ids: List of selected source IDs
""".strip()
```

This prompt serves as a quality gatekeeper, automatically filtering out unreliable sources that could compromise research quality. The key aspect is the structured selection criteria that balance domain reputation, content quality, and relevance. The prompt explicitly targets common quality issues like SEO spam, marketing content, and biased sources that often pollute web search results, ensuring only authoritative sources proceed to the next stage.

### 3.3 Testing the Source Selection Tool

Let's test the source filtering tool to see how it evaluates and selects high-quality sources from our Perplexity results. The tool will analyze each source and provide feedback on which ones meet our quality standards.

In [ ]:
from research_agent_part_2.mcp_server.src.tools import select_research_sources_to_keep_tool

# Test the source selection tool
result = await select_research_sources_to_keep_tool(research_directory=str(research_folder))
print(result)

/Users/jai/Documents/code-repo/agentic-ai-engineering-course/.venv/lib/python3.14/site-packages/firecrawl/v2/types.py:988: UserWarning: Field name "json" in "MonitorPageDiff" shadows an attribute in parent "BaseModel"
  class MonitorPageDiff(BaseModel):
/Users/jai/Documents/code-repo/agentic-ai-engineering-course/.venv/lib/python3.14/site-packages/firecrawl/v2/types.py:1003: UserWarning: Field name "json" in "MonitorPageSnapshot" shadows an attribute in parent "BaseModel"
  class MonitorPageSnapshot(BaseModel):
2026-06-11 20:16:47.509 | INFO     | logging:callHandlers:1737 | Started logging traces to the "nova" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019eb726-65cb-7536-8e44-1007141f2db7&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.
2026-06-11 20:16:47.511 | INFO     | logging:callHandlers:1737 | AFC is enabled with max remote calls: 10.
2026-06-11 20:16:48.536 | INFO     | logging:callHandlers:1737 | HTTP Request: GET https://www.comet.com/

{'status': 'success', 'sources_selected_count': 10, 'selected_source_ids': [16, 17, 19, 20, 21, 22, 46, 53, 54, 55], 'sources_selected_path': '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/.nova/perplexity_sources_selected.md', 'results_selected_path': '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/.nova/perplexity_results_selected.md', 'message': '✅ Selected 10 source(s). IDs written to /Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/.nova/perplexity_sources_selected.md. Filtered results written to /Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/.nova/perplexity_results_selected.md.'}


The tool provides feedback on which sources were selected, allowing users to review the decisions if needed.

## 4. Selecting Sources for Full Content Scraping

After filtering for quality, we need to identify which sources deserve a full scrape. While Perplexity provides summaries and excerpts, some high-quality sources contain much more valuable content in their full form. The `select_research_sources_to_scrape` tool analyzes the filtered results and strategically chooses the most valuable sources for comprehensive content extraction. This full content will provide the writing agent with richer context, detailed examples, and comprehensive coverage that brief excerpts cannot capture.

### 4.1 Understanding the Selection Logic

This tool takes filtered Perplexity results and selects the most valuable sources for full scraping. It analyzes the article guidelines, accepted sources, and already-scraped guideline content to avoid duplication. The tool uses an LLM to evaluate sources based on relevance, authority, quality, and uniqueness, then outputs a prioritized list of URLs. The default limit of 5 sources balances comprehensive coverage with processing efficiency and API costs.

Source: _mcp_server/src/tools/select_research_sources_to_scrape_tool.py_

```python
async def select_research_sources_to_scrape_tool(research_directory: str, max_sources: int = 5) -> Dict[str, Any]:
    """
    Select up to max_sources priority research sources to scrape in full.
    
    Analyzes the filtered Perplexity results together with the article guidelines and
    the material already scraped from guideline URLs, then chooses up to max_sources diverse,
    authoritative sources whose full content will add most value. The chosen URLs are
    written (one per line) to urls_to_scrape_from_research.md.
    """
    # Gather context from the research folder
    guidelines_path = research_path / ARTICLE_GUIDELINE_FILE
    results_selected_path = nova_path / PERPLEXITY_RESULTS_SELECTED_FILE
    
    article_guidelines = read_file_safe(guidelines_path)
    accepted_sources_data = read_file_safe(results_selected_path)
    scraped_guideline_ctx = load_scraped_guideline_context(nova_path)
    
    # Use LLM to select top sources for scraping
    selected_urls, reasoning = await select_top_sources(
        article_guidelines, accepted_sources_data, scraped_guideline_ctx, max_sources
    )
    
    # Write selected URLs to file
    urls_out_path = nova_path / URLS_TO_SCRAPE_FROM_RESEARCH_FILE
    urls_out_path.write_text("\n".join(selected_urls) + "\n", encoding="utf-8")
    
    return {
        "status": "success",
        "sources_selected": selected_urls,
        "sources_selected_count": len(selected_urls),
        "output_path": str(urls_out_path.resolve()),
        "reasoning": reasoning,
        "message": f"Successfully selected {len(selected_urls)} sources for full scraping..."
    }
```

The core of this tool is the `select_top_sources` function, which uses the `PROMPT_SELECT_TOP_SOURCES` prompt to evaluate each source and select the best ones to scrape.

### 4.2 The Source Selection Prompt

The tool uses a prompt to choose the most valuable sources:

```python
PROMPT_SELECT_TOP_SOURCES = """
You are assisting with research for an upcoming article.

Your task is to select the most relevant and trustworthy sources from the web search results.
You should consider:
1. **Relevance**: How well each source addresses the article guidelines
2. **Authority**: The credibility and reputation of the source
3. **Quality**: The depth and accuracy of the information provided
4. **Uniqueness**: Sources that provide unique insights not covered by the scraped guideline URLs

Please select the top {top_n} sources that would be most valuable for the article research.

Return your selection with the following structure:
- **selected_urls**: A list of the most valuable URLs to scrape in full, ordered by priority
- **reasoning**: A short explanation summarizing why these specific URLs were chosen
""".strip()
```

This prompt optimizes resource allocation by strategically selecting sources for expensive full-content scraping. The four-dimensional evaluation framework (relevance, authority, quality, uniqueness) ensures maximum research value while avoiding duplication with already-scraped guideline content. The uniqueness criterion is particularly important as it prevents redundant scraping of similar information. The reasoning requirement provides transparency for human oversight and helps identify potential gaps in the selection logic, making the process auditable and improvable.

### 4.3 Testing the Source Selection Tool

Now let's test the source selection tool to see which URLs it chooses for full content scraping. The tool will analyze our filtered sources and select the most valuable ones based on their potential contribution to the final research.

In [5]:
from research_agent_part_2.mcp_server.src.tools import select_research_sources_to_scrape_tool

# Test selecting sources to scrape
result = await select_research_sources_to_scrape_tool(research_directory=str(research_folder), max_sources=3)
print("Selected sources:")
print(result)

2026-06-11 20:16:56.471 | INFO     | logging:callHandlers:1737 | AFC is enabled with max remote calls: 10.
2026-06-11 20:16:58.759 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://www.comet.com/opik/api/v1/private/spans/batch "HTTP/1.1 204 No Content"
2026-06-11 20:16:58.867 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://www.comet.com/opik/api/v1/private/traces/batch "HTTP/1.1 204 No Content"
2026-06-11 20:16:59.294 | INFO     | logging:callHandlers:1737 | HTTP Request: GET https://www.comet.com/is-alive/ping "HTTP/1.1 200 OK"
2026-06-11 20:16:59.837 | INFO     | logging:callHandlers:1737 | 👍 3 sources selected to scrape.


Selected sources:
{'status': 'success', 'urls_selected_count': 3, 'selected_urls': ['https://docs.perplexity.ai/docs/getting-started/pricing', 'https://docs.perplexity.ai/docs/agent-api/output-control', 'https://openai.github.io/openai-agents-python/mcp/'], 'selection_reasoning': "These three sources provide the highest quality and most direct technical insights required by the article guidelines: (1) Perplexity's pricing page outlines the precise request fee model, token costs, and search context sizing needed for Section 8's cost/budget analysis; (2) Perplexity's structured outputs documentation provides the actual JSON schema format and prompt hinting strategies required for Section 5; (3) The OpenAI Agents SDK page explains how server-hosted MCP prompts can dynamically configure agent behavior and logic at runtime, serving as essential technical grounding for the HITL feedback loop in Section 7.", 'urls_output_path': '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/les

## 5. Scraping Selected Research URLs

The `scrape_research_urls_tool` handles the full content extraction from our selected sources. It works very similarly to the guideline URL scraping tool we saw in lesson 16, using Firecrawl for robust web scraping and an LLM for content cleaning. For this reason, we won't go into the details of the scraping process here, and we only provide the code for testing the tool.

In [6]:
from research_agent_part_2.mcp_server.src.tools import scrape_research_urls_tool

# Test scraping the selected research URLs
result = await scrape_research_urls_tool(research_directory=str(research_folder), concurrency_limit=3)
print("Scraping results:")
print(result)

2026-06-11 20:17:01.215 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://api.firecrawl.dev/v2/scrape "HTTP/1.1 200 OK"
2026-06-11 20:17:01.695 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://www.comet.com/opik/api/v1/private/spans/batch "HTTP/1.1 204 No Content"
2026-06-11 20:17:01.748 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/models/gemini-3.5-flash:countTokens "HTTP/1.1 200 OK"
2026-06-11 20:17:01.754 | INFO     | logging:callHandlers:1737 | AFC is enabled with max remote calls: 10.
2026-06-11 20:17:02.574 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://www.comet.com/opik/api/v1/private/spans/batch "HTTP/1.1 204 No Content"
2026-06-11 20:17:02.701 | INFO     | logging:callHandlers:1737 | HTTP Request: POST https://www.comet.com/opik/api/v1/private/traces/batch "HTTP/1.1 204 No Content"
2026-06-11 20:17:03.416 | INFO     | logging:callHandlers:1737 | HTTP Reque

Scraping results:
{'status': 'success', 'urls_processed': 3, 'urls_total': 3, 'original_urls_count': 3, 'deduplicated_count': 0, 'files_saved': 3, 'output_directory': '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/.nova/urls_from_research', 'saved_files': ['pricing-perplexity.md', 'output-control-perplexity.md', 'model-context-protocol-mcp-openai-agents-sdk.md'], 'message': "Processed 3 new URLs from urls_to_scrape_from_research.md in '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs'. Scraped 3/3 web pages."}


## 6. Creating the Final Research File

The `create_research_file_tool` is the final step of our entire workflow. It takes all the accumulated research data and formats it into a comprehensive, well-organized markdown file that serves as input for the writing agent we'll build in the next part of the course.

### 6.1 Understanding the Compilation Process

This tool serves as the final orchestrator, combining all research data into a comprehensive markdown file. It takes the research directory as input and collects content from multiple sources: filtered Perplexity results, scraped guideline sources, code repositories, YouTube transcripts, and additional research sources. The tool organizes everything into collapsible sections and outputs a structured `research.md` file with detailed statistics about the compilation process.

Source: _mcp_server/src/tools/create_research_file_tool.py_

```python
def create_research_file_tool(research_directory: str) -> Dict[str, Any]:
    """
    Generate comprehensive research.md file from all research data.
    
    Combines all research data including filtered Perplexity results, scraped guideline
    sources, and full research sources into a comprehensive research.md file. The file
    is organized into sections with collapsible blocks for easy navigation.
    """
    # Convert to Path object
    article_dir = Path(research_directory)
    nova_dir = article_dir / NOVA_FOLDER
    
    # Collect all research data
    perplexity_results = read_file_safe(nova_dir / PERPLEXITY_RESULTS_SELECTED_FILE)
    
    # Collect scraped sources from guidelines
    scraped_sources = collect_directory_markdowns_with_titles(nova_dir / URLS_FROM_GUIDELINES_FOLDER)
    code_sources = collect_directory_markdowns_with_titles(nova_dir / URLS_FROM_GUIDELINES_CODE_FOLDER)
    youtube_transcripts = collect_directory_markdowns_with_titles(nova_dir / URLS_FROM_GUIDELINES_YOUTUBE_FOLDER)
    
    # Collect full research sources
    additional_sources = collect_directory_markdowns_with_titles(nova_dir / URLS_FROM_RESEARCH_FOLDER)
    
    # Build comprehensive research sections
    research_results_section = build_research_results_section(perplexity_results)
    scraped_sources_section = build_sources_section("Scraped Sources from Guidelines", scraped_sources)
    code_sources_section = build_sources_section("Code Sources from Guidelines", code_sources)
    youtube_section = build_sources_section("YouTube Transcripts from Guidelines", youtube_transcripts)
    additional_section = build_sources_section("Additional Research Sources", additional_sources)
    
    # Combine all sections into final research file
    research_content = combine_research_sections([
        research_results_section,
        scraped_sources_section,
        code_sources_section,
        youtube_section,
        additional_section
    ])
    
    # Write final research file
    research_file_path = article_dir / RESEARCH_MD_FILE
    research_file_path.write_text(research_content, encoding="utf-8")
    
    return {
        "status": "success",
        "markdown_file": str(research_file_path.resolve()),
        "research_results_count": len(extract_perplexity_chunks(perplexity_results)),
        "scraped_sources_count": len(scraped_sources),
        "code_sources_count": len(code_sources),
        "youtube_transcripts_count": len(youtube_transcripts),
        "additional_sources_count": len(additional_sources),
        "message": f"Successfully created comprehensive research file: {research_file_path.name}"
    }
```

No need to go into the details of the code here, as it's standard Python code for file I/O and string manipulation. We'll simply test the tool and see the final output to better understand what the input of the writing agent will be.

### 6.2 The Research File Structure

The final research file `research.md` is organized into collapsible sections for easy navigation. It will look like the following:

```markdown
# Research Results

## Research Results from Web Search
<details>
<summary>Query: [Original Query]</summary>

### Source [1]: [URL]
[Content from source]

### Source [2]: [URL]
[Content from source]
</details>

## Scraped Sources from Guidelines
<details>
<summary>Source: [Filename]</summary>
[Full scraped content]
</details>

## Code Sources from Guidelines
<details>
<summary>Repository: [Repository Name]</summary>
[Repository analysis and code content]
</details>

## YouTube Transcripts from Guidelines
<details>
<summary>Video: [Video Title]</summary>
[Full video transcript]
</details>

## Additional Research Sources
<details>
<summary>Source: [URL]</summary>
[Full scraped research content]
</details>
```

This structure provides comprehensive coverage while remaining navigable for both humans and AI writing agents.

### 6.3 Testing the Research File Creation

Now let's test the final compilation tool to see how it brings together all our research data into a comprehensive, well-structured file. This represents the final step of our entire research workflow.

In [7]:
from research_agent_part_2.mcp_server.src.tools import create_research_file_tool

# Test creating the final research file
result = create_research_file_tool(research_directory=str(research_folder))
print("Research file creation results:")
print(result)

# Read and display a sample of the generated research file
with open(result["markdown_file"], "r") as f:
    content = f.read()
    print("\nFirst 1000 characters of the research file:")
    print(content[:1000] + "...")

Research file creation results:
{'status': 'success', 'markdown_file': '/Users/jai/Documents/code-repo/agentic-ai-engineering-course/lessons/19_final_outputs/research.md', 'research_results_count': 4, 'scraped_sources_count': 7, 'code_sources_count': 0, 'youtube_transcripts_count': 0, 'additional_sources_count': 7, 'message': '✅ Generated research markdown file:\n  - research.md'}

First 1000 characters of the research file:
# Research

## Research Results

<details>
<summary>How does the Model Context Protocol (MCP) enable dynamic, server-hosted prompts for modifying agent behavior without code changes?</summary>

### Source [16]: https://openai.github.io/openai-agents-python/mcp/

Query: How does the Model Context Protocol (MCP) enable dynamic, server-hosted prompts for modifying agent behavior without code changes?

Answer: The OpenAI Agents SDK documentation explains that **MCP servers can provide prompts that dynamically generate agent instructions**, which are then consumed by mo

## 7. Human-in-the-Loop Feedback Integration

As we saw in the previous lesson, it's possible to integrate human feedback into the research agent workflow by simply providing instructions after invoking the MCP prompt. For example, when running the workflow, users can instruct the agent to:

- Show the source IDs selected by `select_research_sources_to_keep` and ask for approval before proceeding.
- Display the URLs chosen by `select_research_sources_to_scrape` and allow modifications.
- Pause after any step for human review and guidance.

This flexibility allows users to maintain control over the research quality while benefiting from the agent's analytical capabilities. Notice that this is possible because of how the MCP prompt is designed and how the inputs/outputs of the MCP tools are structured!

The next section will show you how to run the complete workflow from the MCP client. If you want, you could run it and provide some of the workflow modifications above to the agent and see how it performs!

## 8. Testing the Complete Agent Workflow

Now let's test the complete end-to-end research agent workflow. We'll use the MCP client to run the full workflow and examine the results.

In [ ]:
import sys

from research_agent_part_2.mcp_client.src.client import main as client_main


async def run_client():
    _argv_backup = sys.argv[:]
    sys.argv = ["client"]
    try:
        await client_main()
    finally:
        sys.argv = _argv_backup


# Start client with in-memory server
await run_client()

Once the client is running, you can:

1. **Start the complete workflow**: Type `/prompt/full_research_instructions_prompt` to load the complete research workflow.
2. **Provide the research directory**: Use the path printed by the setup cell: `The research folder is <the path printed by setup>. Run the complete workflow from start to finish.`
3. **Examine the final output**: Check the generated `research.md` file.
4. **Terminate the client**: Type `/quit` after the agent completes all steps.

## 9. Using Cursor with the MCP Server

Our research agent can also be used directly within Cursor IDE through the MCP protocol. The `mcp_server` folder contains a `.mcp.json.sample` file that shows how to configure Cursor to use our research agent.

Here's the content of the `.mcp.json.sample` file:

```json
{
  "mcpServers": {
    "research-agent": {
      "transport": "stdio",
      "command": "uv",
      "args": [
        "--directory",
        "/absolute/path/to/nova/mcp_server",
        "run",
        "-m",
        "src.server",
        "--transport",
        "stdio"
      ]
    }
  }
}
```

To set this up:
1. In Cursor, click on "Cursor" in the top bar, then "Settings", then "Cursor Settings", then "MCP", and finally on "New MCP Server". Cursor will open a `mcp.json` file.
2. Update the `mcp.json` file as shown above. `"research-agent"` is the name of our MCP server that you are connecting Cursor to. You could also have other MCP servers defined in the same file.
3. Save the `mcp.json` file and restart Cursor. Then, make sure that in the "Cursor Settings" page you see the new "research-agent" MCP server and that it's active. If everything is working fine, it should show the amount of MCP tools, prompts and resources available.
4. Now, you can use the research agent directly within Cursor! Open a new chat in the AI Pane and write `/research-agent` to see the available MCP prompts. You can write `/research-agent/full_research_instructions_prompt` and send it in the chat to start the complete research workflow.

This integration allows you to conduct research directly within your development environment, making it easy to incorporate findings into your coding projects.